# Run an Experiment

Config-first workflow:

1. Pick an experiment id from `configs/experiments/`.
2. Call `run_experiment(cfg)`. That's it.

To add a new experiment, drop a YAML into `configs/experiments/` — no code change.
Every run snapshots its fully-resolved config into `data/processed/xgboost_experiments/<id>/config_snapshot.json` so results are reproducible.

In [17]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config
from src.experiments.runner import run_experiment
from src.viz import make_all_plots


In [64]:
# Pick one and load it. Change this single line to switch experiments.
ACTIVE = 'exp_006b_high_trials_raw'
cfg = load_config(ACTIVE)

print(f'Loaded: {cfg.id}')
print(f'  description : {cfg.description}')
print(f'  hypothesis  : {cfg.hypothesis}')
print(f'  sections    : {cfg.run.sections}')
print(f'  trials      : {cfg.xgb.n_trials}')
print(f'  engineered  : {cfg.features.use_engineered}')
print(f'  results dir : {cfg.results_dir}')

Loaded: exp_006b_high_trials_raw
  description : Engineered features, 50 Optuna trials, no target transform. Isolates the effect of a larger hyperparameter search budget while holding all other settings equal to exp_001_baseline. (raw)

  hypothesis  : With only 10 Optuna trials (exp_001), TPE has insufficient warm-up to meaningfully exploit the 8-dimensional search space (max_depth, lr, subsample, colsample, lambda, alpha, min_child_weight, gamma).  Increasing to 50 trials gives the sampler enough exploration to identify genuinely good regularisation settings for the ~120-row training set, with diminishing returns expected beyond this point.  Gains should be most visible for models with many targets (CY5, Williamson) where 10 trials left the tuning under-converged.

  sections    : ['baseline']
  trials      : 50
  engineered  : False
  results dir : /Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/exp_00

## Run

`run_experiment` loads splits, fits physics parameters for every model, and dispatches to each section listed in `cfg.run.sections`. Each run gets a unique directory under `data/processed/xgboost_experiments/` named `{experiment_id}_{YYYYMMDD_HHMMSS}`, so re-running the same config never overwrites previous results. The path is available as `results.results_dir`.

In [65]:
results = run_experiment(cfg)

ACTIVE EXPERIMENT: exp_006b_high_trials_raw
Description : Engineered features, 50 Optuna trials, no target transform. Isolates the effect of a larger hyperparameter search budget while holding all other settings equal to exp_001_baseline. (raw)

Hypothesis  : With only 10 Optuna trials (exp_001), TPE has insufficient warm-up to meaningfully exploit the 8-dimensional search space (max_depth, lr, subsample, colsample, lambda, alpha, min_child_weight, gamma).  Increasing to 50 trials gives the sampler enough exploration to identify genuinely good regularisation settings for the ~120-row training set, with diminishing returns expected beyond this point.  Gains should be most visible for models with many targets (CY5, Williamson) where 10 trials left the tuning under-converged.

Sections    : ['baseline']
Trials      : 50
Engineered  : False
Log numeric : False
Seed        : 42

Loaded splits: train=136 rows, test=42 rows
Built physics datasets for 5 models.

Variant: baseline (log_numeric_

## Inspect results

`results` is an `ExperimentResults`: a dict of `SectionResult` objects keyed by section name. Every section has `model_summary`, `per_target`, and (optionally) `predictions` DataFrames.

In [66]:
for name, section in results.sections.items():
    print(f'--- {name} ---')
    display(section.sorted_by_r2().head())

print('Best (baseline/sweep/loo, first hit):', results.best())

--- baseline ---


,variant,model_name,model_key,train_r2,train_mse,train_rmse,test_r2,test_mse,test_rmse,n_targets
0,baseline,Power Law,power_law,0.750110,388.810514,13.990054,0.464674,741.285503,19.327581,2
1,baseline,Williamson,williamson,0.482110,13744.702609,59.355619,0.329437,19536.194699,70.912534,4
2,baseline,Sisko,sisko,0.695231,278.404978,9.755776,0.310893,525.025184,13.414536,3
3,baseline,Constrained CY,constrained_cy,0.697388,8203.203935,53.049218,0.258498,13193.897027,67.576587,3
4,baseline,5-parameter CY,cy5,0.611502,7100.485164,38.268574,0.087641,11240.256688,48.430936,5


Best (baseline/sweep/loo, first hit): {'model_name': 'Power Law', 'test_r2': 0.4646742400652403, 'test_rmse': 19.327580626067014}


## Plots

In [67]:
if results.results_dir is None:
    print("Run was not persisted (persist=False) — no results directory to plot into.")
else:
    saved = make_all_plots(results, results.results_dir)
    for p in saved:
        print(p)

/Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/exp_006b_high_trials_raw_20260421_161104/baseline_model_comparison.png
/Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/exp_006b_high_trials_raw_20260421_161104/baseline_per_target_r2_heatmap.png
/Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/exp_006b_high_trials_raw_20260421_161104/baseline_train_vs_test_r2.png
/Users/jaidonfroggatt-morton/Documents/Warwick/Dissertation/MOTS.1/msc-dissertation-project-v2/data/processed/xgboost_experiments/exp_006b_high_trials_raw_20260421_161104/baseline_variants.png


## Compare completed experiments

The `experiment_manifest.json` at the top of `data/processed/xgboost_experiments/` accumulates every run across the project.

In [69]:
import json
import pandas as pd

manifest_path = cfg.experiments_root_path / 'experiment_manifest.json'
if manifest_path.exists():
    runs = json.loads(manifest_path.read_text())['runs']
    df = pd.json_normalize(runs)
    cols = ['run_id', 'experiment_id', 'description', 'sections_run',
            'best.model_name', 'best.test_r2', 'duration_seconds', 'created_at_utc', 'notes']
    display(df[[c for c in cols if c in df.columns]])
else:
    print('No manifest yet. Run an experiment first.')

,run_id,experiment_id,description,sections_run,best.model_name,best.test_r2,duration_seconds,created_at_utc,notes
0,exp_006b_high_trials_raw_20260421_161104,exp_006b_high_trials_raw,"Engineered features, 50 Optuna trials, no targ...",[baseline],Power Law,0.464674,221.7,2026-04-21T16:11:04.197287+00:00,
1,exp_008_batch_feature_20260421_160649,exp_008_batch_feature,Engineered features plus Source_Batch as an ex...,[baseline],Power Law,0.460969,78.7,2026-04-21T16:06:49.848647+00:00,
2,exp_007_log_target_high_trials_20260421_160135,exp_007_log_target_high_trials,"Engineered features, log1p target transform, a...",[baseline],Power Law,0.441281,214.2,2026-04-21T16:01:35.644568+00:00,
3,exp_006b_high_trials_raw_20260421_155434,exp_006b_high_trials_raw,"Engineered features, 50 Optuna trials, no targ...",[baseline],Power Law,0.443126,268.8,2026-04-21T15:54:34.946781+00:00,wrong config. engineered features were enabled...
4,exp_006_high_trials_20260421_154752,exp_006_high_trials,"Engineered features, 50 Optuna trials, no targ...",[baseline],Power Law,0.443126,265.9,2026-04-21T15:47:52.075140+00:00,
5,exp_005_log_target_20260421_154534,exp_005_log_target,Engineered features with log1p target transfor...,[baseline],Power Law,0.501829,30.4,2026-04-21T15:45:34.329415+00:00,
6,exp_011_loo_only_raw_20260421_152756,exp_011_loo_only_raw,Leave-one-out generalisability evaluation (row...,[loo],Power Law,0.664901,255.6,2026-04-21T15:27:56.574007+00:00,
7,exp_010_loo_only_20260421_152239,exp_010_loo_only,Leave-one-out generalisability evaluation (row...,[loo],Power Law,0.647151,264.2,2026-04-21T15:22:39.748376+00:00,
8,exp_004_trials_20_raw_20260421_145230,exp_004_trials_20_raw,Higher trials + raw features.,[baseline],Power Law,0.478763,68.5,2026-04-21T14:52:30.448196+00:00,
9,exp_003_raw_only_20260421_145112,exp_003_raw_only,Raw features only (no engineered features).,[baseline],Power Law,0.491317,30.6,2026-04-21T14:51:12.802008+00:00,


## Annotate a run

Add a free-text note to any completed run without re-running it.
Notes are saved into the manifest and each run's `metadata.json`.

```python
from src.experiments.results import annotate_run

annotate_run(
    run_id='<paste run_id from the table above>',
    note='Your observation here — e.g. Power Law best, CY5 overfit.',
    experiments_root=cfg.experiments_root_path,
)
```

Re-run the comparison cell above to see the note in the table.

In [ ]:
from src.experiments.results import annotate_run

# Replace the run_id and note, then run this cell.
# annotate_run(
#     run_id=results.run_id,
#     note='',
#     experiments_root=cfg.experiments_root_path,
# )